In [359]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GINEConv

In [360]:
## 2
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# relativ zu mac oder nvidia gpus
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
# device = torch.device("mps")
device


device(type='cpu')

In [361]:
## 3
DATA_DIR = Path("../data")
PROCESSED_DIR = Path("../data/processed")
MATCHES_PATH = Path(PROCESSED_DIR / "atp_matches_slams_masters.csv")


In [362]:
##4
## clean up data formats and load into a df, one row one match

matches = pd.read_csv(MATCHES_PATH)

matches["tourney_date"] = pd.to_datetime(matches["tourney_date"])



round_order_map = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

matches["round_order"] = matches["round"].map(round_order_map)

matches = matches.sort_values(["tourney_date", "tourney_id", "round_order"]).reset_index(drop= True)


print(matches.shape)
matches.head

(20057, 50)


<bound method NDFrame.head of       tourney_id     tourney_name surface  draw_size tourney_level  \
0       2006-580  Australian Open    Hard      128.0             G   
1       2006-580  Australian Open    Hard      128.0             G   
2       2006-580  Australian Open    Hard      128.0             G   
3       2006-580  Australian Open    Hard      128.0             G   
4       2006-580  Australian Open    Hard      128.0             G   
...          ...              ...     ...        ...           ...   
20052  2024-0352    Paris Masters    Hard       56.0             M   
20053  2024-0352    Paris Masters    Hard       56.0             M   
20054  2024-0352    Paris Masters    Hard       56.0             M   
20055  2024-0352    Paris Masters    Hard       56.0             M   
20056  2024-0352    Paris Masters    Hard       56.0             M   

      tourney_date  match_num  winner_id  winner_seed winner_entry  ...  \
0       2006-01-16         52     103852         31.0 

In [363]:
INTRA_PATH = PROCESSED_DIR / "match_intransitivity_scores_with_levels.csv"

intra = pd.read_csv(INTRA_PATH)
intra["tourney_date"] = pd.to_datetime(intra["tourney_date"])

# Columns that identify the same match in both datasets.
merge_keys = [
    "tourney_id",
    "tourney_date",
    "round",
    "surface",
    "winner_name",
    "loser_name"
]

# We only merge the columns we actually need.
intra_cols = merge_keys + [
    "I_star_context",
    "I_star_direct",
    "intransitivity_level",
    "intransitivity_level_num"
]

matches = matches.merge(
    intra[intra_cols],
    on=merge_keys,
    how="left"
)

# If some Slam matches did not receive an intransitivity label,
# we mark them explicitly instead of leaving NaN.
matches["intransitivity_level"] = matches["intransitivity_level"].fillna("missing")
matches["intransitivity_level_num"] = matches["intransitivity_level_num"].fillna(-2).astype(int)

print(matches["intransitivity_level"].value_counts(dropna=False))

intransitivity_level
missing                7946
high                   3805
medium                 3530
low                    3383
no_context_evidence    1393
Name: count, dtype: int64


In [364]:
##5
bscore_general = pd.read_csv(PROCESSED_DIR / "bscore_snapshots.csv")
bscore_hard = pd.read_csv(PROCESSED_DIR / "bscore_snapshots_hard.csv")
bscore_clay = pd.read_csv(PROCESSED_DIR / "bscore_snapshots_clay.csv")
bscore_grass = pd.read_csv(PROCESSED_DIR / "bscore_snapshots_grass.csv")

## csv's that act like dictionaries for us, having snapshots of the bscores for all players at every point in time, that is relevant


In [365]:
##6
## here we merge all the "dictionaries of the csvs" into a single dataframe for future use

bscore_snapshots = bscore_general.copy()

if bscore_hard is not None:
    bscore_snapshots = bscore_snapshots.merge(
        bscore_hard,
        on=["tourney_id", "round_order", "player"],
        how = "left"
        )
    
else: 
    bscore_snapshots["bscore_hard"] = np.nan

if bscore_clay is not None:
    bscore_snapshots = bscore_snapshots.merge(
        bscore_clay,
        on=["tourney_id", "round_order", "player"],
        how = "left"
    )
else: 
    bscore_snapshots["bscore_clay"] = np.nan

if bscore_grass is not None:
    bscore_snapshots = bscore_snapshots.merge(
        bscore_grass,
        on=["tourney_id", "round_order", "player"],
        how = "left"
        )
else: 
    bscore_snapshots["bscore_grass"] = np.nan

bscore_snapshots.head()

,tourney_id,round_order,player,bscore_general,bscore_hard,bscore_clay,bscore_grass
0,2011-580,1,Adrian Mannarino,0.000219,7.306624e-17,NaN,NaN
1,2011-580,1,Diego Junqueira,0.000046,3.869109e-17,NaN,NaN
2,2011-580,1,Dmitry Tursunov,0.019328,1.402140e-02,NaN,NaN
3,2011-580,1,Fernando Verdasco,0.154029,9.080492e-02,NaN,NaN
4,2011-580,1,Marc Gicquel,0.012592,5.609831e-03,NaN,NaN


In [366]:
##7
for col in ["bscore_hard", "bscore_clay", "bscore_grass"]:
    bscore_snapshots[col] = bscore_snapshots[col].fillna(bscore_snapshots["bscore_general"])

bscore_snapshots.isna().sum()

tourney_id        0
round_order       0
player            0
bscore_general    0
bscore_hard       0
bscore_clay       0
bscore_grass      0
dtype: int64

In [367]:
##8
def surface_one_hot(surface):
    if surface == "Hard":
        return[1.0, 0.0, 0.0]
    elif surface =="Clay":
        return[0.0, 1.0, 0.0]
    elif surface == "Grass":
        return[0.0, 0.0, 1.0]
    else:  return [0.0, 0.0, 0.0]

In [368]:
##9
def relative_game_diff(score_str):
    if pd.isna(score_str):
        return np.nan
    
    total_winner_games = 0
    total_loser_games = 0

    total_games = 0

    sets = str(score_str).split()

    for s in sets:
        s = re.sub(r"\(.*?\)", "", s)

        if re.match(r"^\d+-\d+$", s):
            a, b = s.split("-")
            total_winner_games += int(a)
            total_loser_games += int(b)

    total_games = total_winner_games + total_loser_games

    if total_games == 0:
        return np.nan
    else: 
        return ((total_winner_games -total_loser_games ) / total_games)

In [369]:
##10
## a dataframe of all the players with no nans

## alle spieler die als gewinner im Dataset sind
winner_static_matches = matches[
    ["winner_name", "winner_ht", "winner_hand", "tourney_date"]
].rename(columns={"winner_name": "player",
                  "winner_ht": "height",
                  "winner_hand": "hand"})

## alle loser im Dataset
loser_static_matches = matches[
    ["loser_name", "loser_ht", "loser_hand", "tourney_date"]
].rename(columns={"loser_name": "player",
                  "loser_ht": "height",
                  "loser_hand": "hand"})

## beide df zusammen
player_static_df = pd.concat([winner_static_matches, loser_static_matches], ignore_index=True)
player_static_df = (
    player_static_df
    .sort_values("tourney_date")
    .groupby("player")
    .last()
)

## median der höhe der spieler um nan zu füllen
median_height = player_static_df["height"].dropna().median()
player_static_df["height"] = player_static_df["height"].fillna(median_height)

## eher rechts als linkshändler
player_static_df["hand"] = player_static_df["hand"].fillna("R")

player_static_df.head()

,height,hand,tourney_date
player,,,
Abedallah Shelbayh,180.0,L,2023-04-24
Adam Pavlasek,185.0,R,2018-05-28
Adam Walton,183.0,R,2024-08-26
Adrian Mannarino,180.0,L,2024-10-28
Adrian Menendez Maceiras,185.0,R,2017-08-28


In [370]:
## diese cell konstruiert die rolling blocks damit wir sie nicht im training loop machen müssen
## wir brauchen als Kontext auch die Matches pre Training Loop, ansonsten sind die embeddings viel zu 
# "schlecht"

START_DATE = "2006-01-01"

TRAIN_END = 2015
VAL_END = 2016

## future matches sind wo training beginnen wird
future_match = matches[matches["tourney_date"] >= "2011-01-01"].copy()
future_matches = future_match[future_match["tourney_date"] < "2021-01-01"].copy()

## tabelle mit rows tipo: Australian Open 2011 1
## drop duplicates weil viele same round matches in einem Turnier logisch sind
rolling_blocks = (
    future_matches[["tourney_date", "tourney_id", "round_order"]]
    .drop_duplicates()
    .sort_values(["tourney_date", "tourney_id", "round_order"])
    .reset_index(drop = True)
)

def assign_phase(year):
    if year <= 2010:
        return "warmup"
    elif year <= TRAIN_END:
        return "train"
    elif year <= VAL_END:
        return "val"
    else:
        return "test"

## redundant technically
# rolling_blocks = rolling_blocks[
#     rolling_blocks["tourney_date"].dt.year >= 2006
# ].reset_index(drop = True)

## 2 new columns
rolling_blocks["year"] = rolling_blocks["tourney_date"].dt.year
rolling_blocks["phase"] = rolling_blocks["tourney_date"].dt.year.apply(assign_phase)

print(rolling_blocks["phase"].value_counts())
rolling_blocks.head()

## Konstruktion rolling blocks
# jedes block eine runde
# im training arbeiten wir immer nur am current block
# graph is gebaut von training blocks und der history, und nichts was noch nicht geschehen ist

phase
train    420
test     291
val       84
Name: count, dtype: int64


,tourney_date,tourney_id,round_order,year,phase
0,2011-01-17,2011-580,1,2011,train
1,2011-01-17,2011-580,2,2011,train
2,2011-01-17,2011-580,3,2011,train
3,2011-01-17,2011-580,4,2011,train
4,2011-01-17,2011-580,5,2011,train


In [371]:
SCORE_COLS = [
    "bscore_general",
    "bscore_hard",
    "bscore_clay",
    "bscore_grass"
]

bscore_index = {}
for (t_id, r_ord), group in bscore_snapshots.groupby(["tourney_id", "round_order"]):
    bscore_index[(t_id, r_ord)] = group.set_index("player")[SCORE_COLS].to_dict("index")


## die funktion gibt die bscore snapshots zurück indem man sie auf die Informationen des rolling_blocks aufruft
## es berechnet auch die defaults wenn vielleicht bei einem spieler die values mssing sind
def get_bscore_snapshots(tourney_id, round_order):
    snap_dict = bscore_index.get((tourney_id, round_order), {})

    ## passiert nicht aber good practice ig
    # missing_cols = [col for col in SCORE_COLS if col not in snap.columns]
    # if missing_cols:
    #     raise ValueError(f"Missing bscore columns: {missing_cols}")

    ## wenn der snapshot nicht existiert (ist nie der Fall) dann ist bscore 0.0 für alle, 
    # vielleicht median über ganzes bscore besser als 0.0
    # if snap.empty:
    #     defaults = {col: 0.0 for col in SCORE_COLS}
    #     return {}, defaults

    defaults = {}
    for col in SCORE_COLS:
        values = [
            float(v[col])
            for v in snap_dict.values()
            if col in v and not pd.isna(v[col])
        ]
        if values:
            defaults[col] = float(np.quantile(values, 0.25))
        else:
            defaults[col] = 0.0 

    # all_vals = {col: [v.get(col, 0.0) for v in snap_dict.values()] for col in SCORE_COLS}
    # defaults = {col: float(np.quantile(all_vals[col], 0.25)) if all_vals[col] else 0.0 for col in SCORE_COLS}
    return snap_dict, defaults
    # spieler und seine bscore werden zusammen im dictionary gespeichert

## für jedes block nehmen wir die bscore die wir bereits im vorherein berechnet haben und returnen nur diese und default

In [372]:
## wir builden den GNN graph eines einzelnen blocks
## als input geben wir die history bis zu block, die matches des blocks, und alle bscore vor den matches
## las output bekommen wir den "data" obj von PyG und ein player_idx um ein mapping von name zu indizes zu haben
## kein leakage von den matches innerhalb des blocks weil graph auf die alten gebaut wird, dann prediction -> loop

def build_pyg_snapshot(current_history, block_matches, t_block, snap_dict, defaults, alpha_days=365):

    MAX_HISTORY_DAYS = 365 * 3  # usa solo ultimi 3 anni
    cutoff = t_block - pd.Timedelta(days=MAX_HISTORY_DAYS)
    current_history_trimmed = current_history[current_history["tourney_date"] >= cutoff]
    ## alle spieler die in dem block bis jetzt auftauchen
    all_players = pd.Index(
        pd.unique(
            pd.concat(
                [
                    current_history_trimmed["winner_name"],
                    current_history_trimmed["loser_name"],
                    block_matches["winner_name"],
                    block_matches["loser_name"],
                ],
                ignore_index=True
            )
        )
    )

    ## für alle spieler anstatt namen auch eine Zahl, damit man nicht mit strings arbeitet
    player_to_idx = {player: i for i, player in enumerate(all_players)}

    node_features = []

    ## dieses Loop nimmt die features von jedem Spieler aktualisiert auf Zeitpunkt
    ## und buildet dann die tensoren damit für die GNN
    for player in all_players:
        scores = snap_dict.get(player, defaults)

        bsg = float(scores.get("bscore_general", defaults["bscore_general"]))
        bsh = float(scores.get("bscore_hard", defaults["bscore_hard"]))
        bsc = float(scores.get("bscore_clay", defaults["bscore_clay"]))
        bsgra = float(scores.get("bscore_grass", defaults["bscore_grass"]))

        if player in player_static_df.index:
            height = player_static_df.loc[player, "height"]
            hand = player_static_df.loc[player, "hand"]
        else:
            height = median_height
            hand = "R"

        if pd.isna(height):
            height = median_height

        
        is_right = 1.0 if hand == "R" else 0.0

        node_features.append([
            bsg,
            bsh,
            bsc,
            bsgra,
            height,
            is_right
        ])

    x = torch.tensor(node_features, dtype=torch.float)

    edge_rows = []

    ## hier builden wir die arcs zwischen den Spieler die von den Matches der current history genommen werden, also auch alte
    ## die edges gehen immer von verliere zu gewinner damit sie ähnlich zum bscore graph sind zur Kohärenz
    ## wir speichern im edge wie alt das match ist(es sind auch die alten hier dabei)
    ## hier iteration über alle rows nicht die spieler
    for _, row in current_history_trimmed.iterrows():
        loser = row["loser_name"]
        winner = row["winner_name"]

        if loser not in player_to_idx or winner not in player_to_idx:
            continue

        src = player_to_idx[loser]
        dst = player_to_idx[winner]

        winner_scores = snap_dict.get(winner, defaults)
        loser_scores = snap_dict.get(loser, defaults)

        winner_bscore = float(
        winner_scores.get(
            "bscore_general",
            defaults["bscore_general"]
        )
        )

        loser_bscore = float(
            loser_scores.get(
                "bscore_general",
                defaults["bscore_general"]
            )
        )

        winner_surface_bscore = get_surface_bscore(
            winner_scores,
            row["surface"]
        )

        loser_surface_bscore = get_surface_bscore(
            loser_scores,
            row["surface"]
        )

        bscore_diff_general = winner_bscore - loser_bscore

        bscore_diff_surface = (
            winner_surface_bscore
            - loser_surface_bscore
        )

        age_days = (t_block - row["tourney_date"]).days

        ## sollte nicht passieren
        if age_days < 0:
            continue

        ## best case 1, tendiert zu 0, alpha days kann überarbeitet werden
        recency_weight = 1.0 / (1.0 + (age_days / alpha_days))

        ## wie Stark war der Sieg relativ/wie dominant
        rel_gd = relative_game_diff(row["score"])
        if pd.isna(rel_gd):
            rel_gd = 0.0

        ## scaling von round order zwischen 0-1
        round_ord = row["round_order"]
        round_scaled = float(round_ord) / 7.0 if pd.notna(round_ord) else 0.0

        ## shape der edges ist also 6 features
        edge_features = ([recency_weight, rel_gd] + surface_one_hot(row["surface"]) + [round_scaled])

        edge_rows.append([src, dst] + edge_features)

    if len(edge_rows) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 8), dtype=torch.float)
    else:
        ## pytorch als eine matrix 2xnum_edges wo die sources oben sind und dst unten
        edge_index = torch.tensor(
            [[row[0], row[1]] for row in edge_rows],
            dtype=torch.long
        ).t().contiguous()

        ## nur die features genommen hier
        edge_attr = torch.tensor(
            [row[2:] for row in edge_rows],
            dtype=torch.float
        )

    ## NN's wissen nicht was zu tun mit nan's also im fall auf 0 setzen
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    edge_attr = torch.nan_to_num(edge_attr, nan=0.0, posinf=0.0, neginf=0.0)

    ## das eigentliche Graph was für den message passing gebraucht ist
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr
    )

    return data, player_to_idx

In [373]:
def get_surface_bscore(scores, surface):
    if surface == "Hard":
        return float(scores.get("bscore_hard", scores.get("bscore_general", 0.0)))
    elif surface == "Clay":
        return float(scores.get("bscore_clay", scores.get("bscore_general", 0.0)))
    elif surface== "Grass":
        return float(scores.get("bscore_grass", scores.get("bscore_general", 0.0)))
    else:
        return float(scores.get("bscore_general", 0.0))
        
## diese funktion soll uns die targets randomizen, damit man nicht immer winner als player1 hat

## nimmt die current matches und wandelt sie in ein tensor um fürs modell benutzbar zu sein
## praktisch graph gibt es schon das sind die examples fürs Training
## paar lines sind von experimente die man nicht braucht
def make_block_targets(block_matches, player_to_idx, rng, snap_dict, defaults):

    rows = []

    for _, row in block_matches.iterrows():
        winner = row["winner_name"]
        loser= row["loser_name"]

        if winner not in player_to_idx or loser not in player_to_idx: 
            continue

        winner_idx = player_to_idx[winner]
        loser_idx = player_to_idx[loser]

        winner_scores = snap_dict.get(winner, defaults)
        loser_scores = snap_dict.get(loser, defaults)

        winner_bscore = float(winner_scores.get("bscore_general", defaults["bscore_general"]))
        loser_bscore = float(loser_scores.get("bscore_general", defaults["bscore_general"]))

        match_surface = surface_one_hot(row["surface"])

        round_ord = row["round_order"]
        ## scaling damit die daten zwischen 0 und 1 liegen
        round_scaled = float(round_ord) / 7.0 if pd.notna(round_ord) else 0.0


        if rng.random() < 0.5:
            player_a_idx = winner_idx
            player_b_idx = loser_idx
            y = 1.0
            bscore_diff = winner_bscore - loser_bscore
        else:
            player_a_idx = loser_idx
            player_b_idx = winner_idx
            y = 0.0
            bscore_diff = loser_bscore - winner_bscore

        match_context = match_surface + [round_scaled]
        

        rows.append({
            "player_a_idx": player_a_idx,
            "player_b_idx": player_b_idx,
            "match_context": match_context,
            "y": y,
            "bscore_diff": bscore_diff,
            "intransitivity_level": row["intransitivity_level"]
        })

    if len(rows) == 0:
        return None
    
    player_a = torch.tensor([r["player_a_idx"] for r in rows], dtype=torch.long)
    player_b = torch.tensor([r["player_b_idx"] for r in rows], dtype=torch.long)
    context = torch.tensor([r["match_context"] for r in rows], dtype=torch.float)
    y = torch.tensor([r["y"] for r in rows], dtype=torch.float)
    bscore_diffs = torch.tensor([r["bscore_diff"] for r in rows], dtype=torch.float)
    intransitivity_levels = [
        r["intransitivity_level"]
        for r in rows
    ]

    return player_a, player_b, context, y, bscore_diffs, intransitivity_levels

In [374]:
## die GNN macht die embeddings der Spieler
## wir nehmen embedding von a, b difference und context des matches produzieren ein logit
## das logit wir dann probbaility durch sigmoid

## node features Infos über spieler
## edge features infos über matches
## match context infos über das zu predictende Match
## output ist dann eine probability

## wir haben also eigentlich 3 NN's
## node_encoder ist eine lineare NN was nur die feature in der internen dimension des GNN bringt
## nn1 und nn2, sind die nn's was GINE braucht für das Message Passing, eine pro Layer
## ein Layer arbeitet auf direkte nicht bearbeite relations, nn2 auf schon überarbeitete infos
## match_head ist die Finale NN was die eigentliche prediction macht 
class TennisGINE(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim, match_context_dim, dropout=0.5):
        super().__init__()
        ## constructor
        ## self, node_i_d(how many features does a player have), e_i_d(anzahl feature der edges)
        ## hidd_dim(interene rappresentation des spielers als Vektor), match_c_d(dimension der feature des matches)

        self.node_encoder = nn.Linear(node_in_dim, hidden_dim)
        ## das bringt die Nodes in der Dimension das unsere GNN benutzt (project features auf hidden dimensions)

        nn1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        ## kleine interne NN, braucht GINE um die Infos der neighbors zu transformieren

        nn2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        ## wie oben, sind 2 weil die layer zwei transformationen lernen, eine mehr in depth der anderen

        self.conv1 = GINEConv(nn1, edge_dim=edge_in_dim, aggr="sum")
        self.conv2 = GINEConv(nn2, edge_dim=edge_in_dim, aggr="sum")
        ## das sind die 2 GINEConv Layers, also die effektive GNN
        ## conv1 auf nur neighbors
        ## conv2 auf neighbors der neighbors auch
        ## sum aggregation besser als mean muss aber wieder mit tuning getestet werden

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        ## diese Layer stabilisieren die embedding, normalisierung damit wir enorme zu kleine und instabile daten haben

        pair_dim = hidden_dim * 4
        ## pair_z hat 4 embeddings drin..

        ## feed forward NN
        self.match_head = nn.Sequential(
            nn.Linear(pair_dim + match_context_dim, hidden_dim), ## combine die embeddings der spieler
            nn.ReLU(), ## nicht linearität
            nn.Dropout(dropout), ## damit man nicht overfitting macht(dataset nicht enorm) teil der activations werden "rausgeschmissen"
            nn.Linear(hidden_dim, hidden_dim // 2), ## comprimierung der Infos richtung binary decision
            nn.ReLU(), ## ...
            nn.Dropout(dropout), ## ..
            nn.Linear(hidden_dim // 2, 1) ## produces eine einzige Zahl was Logit ist
        )
        ## das ist eine letzte NN die die Matches vorhersagt
        ## so werden embeddings und match context prozessiert für die Vorhersage


    def forward(self, data, player_a_idx, player_b_idx, match_context):
        ## diese Funktion definiert was passiert wenn Infos ins Modell kommen

        h = self.node_encoder(data.x)
        ## feature des spielers ins embedding, current representation der spieler

        h = self.conv1(h, data.edge_index, data.edge_attr)
        h = self.norm1(F.relu(h))
        ## update des spielers durch embedding und neighbors, edge features
        ## F.relu brauchen wir um nicht linearität im Modell einzuführen

        h = self.conv2(h, data.edge_index, data.edge_attr)
        h = self.norm2(F.relu(h))
        ## spieler 2 bekommt infos auch von den bereits updated neighbors

        ## mit 2 layer können wir sowohl direct relations als auch indirekte einbauen

        h_a = h[player_a_idx]
        h_b = h[player_b_idx]

        pair_z = torch.cat(
            [
                h_a,
                h_b,
                h_a - h_b,
                torch.abs(h_a - h_b)
            ],
            dim=1
        )
        z = torch.cat([pair_z, match_context], dim=1)

        logits = self.match_head(z).squeeze(-1)

        ## wichtig: weil match_head eine NN mit ReLU und bias ist, ist nicht sicher dass P(A)(A gewinnt) und P(B)(B gewinnt)
        ## komplementär sind, das kann im Fall noch implementieren, so lernt die "Kohärenz" der P(X)-P(Y) selber
        ## was wir returnen ist nicht direkt eine Probability, sondern direkt ein Value, was die GNNs benutzen können 
        ## um im Training BCEWithLogitsLoss zu benutzen, was besser ist (laut Dokumentation)

        return logits, (h_a - h_b).detach()

In [375]:
model = TennisGINE(
    node_in_dim=6,
    edge_in_dim=6,
    hidden_dim= 128,
    match_context_dim=4,
    dropout = 0.5
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

model

TennisGINE(
  (node_encoder): Linear(in_features=6, out_features=128, bias=True)
  (conv1): GINEConv(nn=Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  ))
  (conv2): GINEConv(nn=Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  ))
  (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (match_head): Sequential(
    (0): Linear(in_features=516, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [376]:
print(bscore_snapshots.columns.tolist())
print(bscore_snapshots.head())

['tourney_id', 'round_order', 'player', 'bscore_general', 'bscore_hard', 'bscore_clay', 'bscore_grass']
  tourney_id  round_order             player  bscore_general   bscore_hard  \
0   2011-580            1   Adrian Mannarino        0.000219  7.306624e-17   
1   2011-580            1    Diego Junqueira        0.000046  3.869109e-17   
2   2011-580            1    Dmitry Tursunov        0.019328  1.402140e-02   
3   2011-580            1  Fernando Verdasco        0.154029  9.080492e-02   
4   2011-580            1       Marc Gicquel        0.012592  5.609831e-03   

   bscore_clay  bscore_grass  
0     0.000219      0.000219  
1     0.000046      0.000046  
2     0.019328      0.019328  
3     0.154029      0.154029  
4     0.012592      0.012592  


In [377]:
rng = np.random.default_rng(SEED)

ROLLING_START_DATE = "2011-01-01"

current_history = matches[
    (matches["tourney_date"] <= "2011-01-01")
].copy()

logs = []
pred_rows = []

## wie viele optimizer updates nach jedem Block
steps_per_block = 2

## blocks für stabilisation des online trainings
replay_buffer = []
replay_buffer_size = 200

WARMUP_END = 2011
UPDATE_PHASES =  {"train"}



for block_idx, block in rolling_blocks.iterrows():
    tourney_id = block["tourney_id"]
    round_order = block["round_order"]
    t_block = block["tourney_date"]
    phase = block["phase"]
    year = block["year"]

    block_matches = future_matches[
        (future_matches["tourney_id"] == tourney_id) &
        (future_matches["round_order"] == round_order)
    ].copy()

    block_matches_all = block_matches.copy()

    # During train and validation, use all matches.
    # During test, only evaluate high-intransitivity matches.
    block_matches_target = block_matches.copy()


    snap_dict, defaults = get_bscore_snapshots(tourney_id, round_order)

    ## graph snapshot mit nur past matches
    data,player_to_idx = build_pyg_snapshot(
        current_history=current_history, 
        block_matches= block_matches_all, 
        t_block=t_block, 
        snap_dict=snap_dict, 
        defaults=defaults, 
        alpha_days=365)

    target_tensors = make_block_targets(block_matches=block_matches_target, player_to_idx=player_to_idx, snap_dict = snap_dict, rng=rng, defaults = defaults)

    if target_tensors is None:
        current_history = pd.concat([current_history, block_matches_all], ignore_index=True)
        continue

    player_a, player_b, match_context, y, bscore_diffs, intransitivity_levels = target_tensors

    data= data.to(device)
    player_b = player_b.to(device)
    player_a = player_a.to(device)
    match_context = match_context.to(device)
    y = y.to(device)

    ## evaluate bevor wir auf dem Block trainieren
    model.eval()
    with torch.no_grad():
        logits_eval, emb_diff_tensor = model(data, player_a, player_b, match_context)
        emb_diff = emb_diff_tensor.cpu().numpy()
        probs_eval = torch.sigmoid(logits_eval)
        loss_before = criterion(logits_eval, y).item()
        preds = (probs_eval >= 0.5).float()
        acc_before = (preds == y).float().mean().item()

    probs_np = probs_eval.cpu().numpy()
    preds_np = preds.cpu().numpy()
    y_np = y.cpu().numpy()

    for i in range(len(y_np)):
        pred_rows.append({
            "tourney_id": tourney_id,
            "round_order": round_order,
            "tourney_date": t_block,
            "phase": phase,
            "year": year,
            "prob_before": float(probs_np[i]),
            "pred_before": float(preds_np[i]),
            "y_true": float(y_np[i]),
            "block_idx": block_idx,
            "emb_diff": emb_diff[i].tolist(),
            "bscore_diff": float(bscore_diffs[i].item()),
            "intransitivity_level": intransitivity_levels[i]
        })

    if phase in UPDATE_PHASES and year > WARMUP_END:

        ## kopie des blocks im replay buffer
        data_buf = data.clone()
        data_buf.x = data_buf.x.detach()
        data_buf.edge_index = data_buf.edge_index.detach()
        data_buf.edge_attr = data_buf.edge_attr.detach()

        replay_buffer.append((
            data_buf,
            player_a.detach(),
            player_b.detach(),
            match_context.detach(),
            y.detach()
        ))

        if len(replay_buffer) > replay_buffer_size:
            replay_buffer = replay_buffer[-replay_buffer_size:]

        model.train()

        for _ in range(steps_per_block):
            optimizer.zero_grad()

            losses = []

            for replay_data, replay_a, replay_b, replay_context, replay_y in replay_buffer:
                logits, _ = model(
                    replay_data,
                    replay_a,
                    replay_b,
                    replay_context
                )

                loss = criterion(logits, replay_y)
                losses.append(loss)

            total_loss = torch.stack(losses).mean()

            total_loss.backward()
            optimizer.step()

        ##evaluation nach dem update
        model.eval()

        with torch.no_grad():
            logits_after, _ = model(data, player_a, player_b, match_context)
            probs_after = torch.sigmoid(logits_after)

            loss_after = criterion(logits_after, y).item()
            preds_after = (probs_after >= 0.5).float()
            acc_after = (preds_after == y).float().mean().item()
    else:
        loss_after = np.nan
        acc_after = np.nan


    ## die after stats sagen uns eher ob die GNN gerade irgendwas macht oder nicht, sie sind nicht metriken die uns viel sagen
    logs.append({
        "block_idx": block_idx,
        "tourney_id": tourney_id,
        "phase": phase,
        "year": year,
        "round_order": round_order,
        "num_matches": len(block_matches),
        "loss_before": loss_before,
        "acc_before": acc_before,
        "loss_after": loss_after,
        "acc_after": acc_after,
        "num_nodes": data.num_nodes,
        "num_edges": data.num_edges
    })
    
    current_history = pd.concat([current_history, block_matches_all], ignore_index=True)

    if block_idx %20 == 0:
        print(
            f"({block_idx:03d} {tourney_id} {block['round_order']}) "
            f"loss_before={loss_before:.4f} acc_before={acc_before:.3f} "
            f"loss_after={loss_after:.4f} acc_after={acc_after:.3f}"
        )
    

(000 2011-580 1) loss_before=0.6990 acc_before=0.484 loss_after=nan acc_after=nan
(020 2011-403 7) loss_before=0.6364 acc_before=1.000 loss_after=nan acc_after=nan
(040 2011-520 2) loss_before=0.6898 acc_before=0.531 loss_after=nan acc_after=nan
(060 2011-422 3) loss_before=0.7081 acc_before=0.375 loss_after=nan acc_after=nan
(080 2011-352 4) loss_before=0.7090 acc_before=0.375 loss_after=nan acc_after=nan
(100 2012-403 3) loss_before=0.6738 acc_before=0.625 loss_after=0.6642 acc_after=0.750
(120 2012-416 5) loss_before=0.4788 acc_before=0.750 loss_after=0.4801 acc_after=0.750
(140 2012-421 5) loss_before=0.5989 acc_before=0.500 loss_after=0.5925 acc_after=0.500
(160 2012-5014 6) loss_before=0.5498 acc_before=1.000 loss_after=0.5306 acc_after=1.000
(180 2013-404 6) loss_before=1.4744 acc_before=0.500 loss_after=1.4050 acc_after=0.500
(200 2013-1536 7) loss_before=0.1117 acc_before=1.000 loss_after=0.1007 acc_after=1.000
(220 2013-540 7) loss_before=1.4260 acc_before=0.000 loss_after=1.

In [378]:
logs_df = pd.DataFrame(logs)
predictions_df = pd.DataFrame(pred_rows)

logs_df.head()

,block_idx,tourney_id,phase,year,round_order,num_matches,loss_before,acc_before,loss_after,acc_after,num_nodes,num_edges
0,0,2011-580,train,2011,1,64,0.698973,0.484375,NaN,NaN,304,3090
1,1,2011-580,train,2011,2,32,0.681009,0.593750,NaN,NaN,304,3154
2,2,2011-580,train,2011,3,16,0.686048,0.562500,NaN,NaN,304,3186
3,3,2011-580,train,2011,4,8,0.708381,0.375000,NaN,NaN,304,3202
4,4,2011-580,train,2011,5,4,0.694506,0.500000,NaN,NaN,304,3210


In [379]:


def phase_metrics(df):
    y = df["y_true"].to_numpy()
    p = df["prob_before"].to_numpy()
    pred = df["pred_before"].to_numpy()

    eps = 1e-7
    p = np.clip(p, eps, 1- eps)
    
    accuracy = (pred==y).mean()
    logloss= -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
    brier= np.mean((p-y)**2)
    # auc = roc_auc_score(y, p)

    return pd.Series({
        "n_matches": len(df),
        "accuracy": accuracy,
        "logloss" : logloss,
        "brier": brier,
        # "auc": auc
    })

metrics_by_phase = (
    predictions_df
    .groupby("phase")
    .apply(phase_metrics)
)

metrics_by_phase


,n_matches,accuracy,logloss,brier
phase,,,,
test,3770.0,0.659947,0.631730,0.217822
train,5367.0,0.662195,0.596825,0.205822
val,1075.0,0.700465,0.571368,0.193055


In [380]:
test_predictions = predictions_df[
    predictions_df["phase"] == "test"
].copy()

metrics_by_intransitivity = (
    test_predictions
    .groupby("intransitivity_level")
    .apply(phase_metrics)
    .reindex(["low", "medium", "high"])
)

metrics_by_intransitivity

,n_matches,accuracy,logloss,brier
intransitivity_level,,,,
low,857.0,0.668611,0.623726,0.213927
medium,914.0,0.665208,0.643237,0.221561
high,1048.0,0.665076,0.624037,0.214387


In [381]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score


bscore_base = pd.read_csv(PROCESSED_DIR / "atp_matches_with_bscore_and_intransitivity.csv")

bscore_base["tourney_date"] = pd.to_datetime(bscore_base["tourney_date"])

categories = ["G", "M"]

bscore_base = bscore_base[
    (bscore_base["tourney_date"] >= START_DATE)
    & (bscore_base["tourney_level"].isin(categories))
].copy()

bscore_base = bscore_base.dropna(subset=[
    "winner_score_pre",
    "loser_score_pre",
    "tourney_date"
]).copy()

bscore_base["year"] = bscore_base["tourney_date"].dt.year
bscore_base["phase"] = bscore_base["year"].apply(assign_phase)

# raw diff: winner bscore - loser bscore
bscore_base["winner_loser_bscore_diff"] = (
    bscore_base["winner_score_pre"] - bscore_base["loser_score_pre"]
)

rng = np.random.default_rng(SEED)

# True = player A is the winner
a_is_winner = rng.random(len(bscore_base)) < 0.5

bscore_base["x_bscore_diff"] = np.where(
    a_is_winner,
    bscore_base["winner_loser_bscore_diff"],
    -bscore_base["winner_loser_bscore_diff"]
)

bscore_base["y"] = np.where(a_is_winner, 1, 0)

bscore_base[[
    "tourney_date",
    "phase",
    "winner_name",
    "loser_name",
    "winner_score_pre",
    "loser_score_pre",
    "x_bscore_diff",
    "y",
    "intransitivity_level"
]].head()

,tourney_date,phase,winner_name,loser_name,winner_score_pre,loser_score_pre,x_bscore_diff,y,intransitivity_level
0,2011-01-17,train,Roger Federer,Lukas Lacko,0.435933,0.006462,-0.429471,0,low
1,2011-01-17,train,Xavier Malisse,Pablo Andujar,0.027134,0.003305,0.023829,1,no_context_evidence
2,2011-01-17,train,Marcos Baghdatis,Grega Zemlja,0.094813,0.002176,-0.092637,0,low
3,2011-01-17,train,Juan Martin del Potro,Dudi Sela,0.126760,0.016866,-0.109895,0,medium
4,2011-01-17,train,Pere Riba,Carsten Ball,0.004381,0.001403,0.002977,1,no_context_evidence


In [382]:
train_b = bscore_base[bscore_base["phase"] == "train"].copy()
val_b = bscore_base[bscore_base["phase"] == "val"].copy()
test_b = bscore_base[bscore_base["phase"] == "test"].copy()

print(len(train_b), len(val_b), len(test_b))

4617 925 6569


In [383]:
mu = train_b["x_bscore_diff"].mean()
sigma = train_b["x_bscore_diff"].std()

if sigma == 0 or np.isnan(sigma):
    sigma = 1.0

for df in [train_b, val_b, test_b]:
    df["x_bscore_diff_scaled"] = (df["x_bscore_diff"] - mu) / sigma

In [384]:
test_b_gnn_subset = predictions_df[predictions_df["phase"] == "test"].copy()

test_b_gnn_subset["x_bscore_diff"] = test_b_gnn_subset["bscore_diff"]
test_b_gnn_subset["y"] = test_b_gnn_subset["y_true"]

test_b_gnn_subset["x_bscore_diff_scaled"] = (
    test_b_gnn_subset["x_bscore_diff"] - mu
) / sigma

print("GNN test rows:", len(test_b_gnn_subset))


GNN test rows: 3770


In [385]:
def evaluate_bscore_rule(df):
    y_true = df["y"].to_numpy()
    y_pred = (df["x_bscore_diff"] >= 0).astype(int)

    return pd.Series({
        "n_matches": len(df),
        "accuracy": accuracy_score(y_true, y_pred)
    })

rule_results = pd.DataFrame({
    "train": evaluate_bscore_rule(train_b),
    "val": evaluate_bscore_rule(val_b),
    "test": evaluate_bscore_rule(test_b_gnn_subset)
}).T

rule_results

,n_matches,accuracy
train,4617.0,0.711068
val,925.0,0.715676
test,3770.0,0.644562


In [386]:
X_train = train_b[["x_bscore_diff_scaled"]]
y_train = train_b["y"]

bscore_logit = LogisticRegression()
bscore_logit.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [387]:
def evaluate_prob_model(model, df):
    X = df[["x_bscore_diff_scaled"]]
    y_true = df["y"].to_numpy()

    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)

    return pd.Series({
        "n_matches": len(df),
        "accuracy": accuracy_score(y_true, pred),
        "log_loss": log_loss(y_true, prob),
        "brier": brier_score_loss(y_true, prob),
        "auc": roc_auc_score(y_true, prob)
    })

bscore_logit_results = pd.DataFrame({
    "train": evaluate_prob_model(bscore_logit, train_b),
    "val": evaluate_prob_model(bscore_logit, val_b),
    "test": evaluate_prob_model(bscore_logit, test_b_gnn_subset)
}).T

bscore_logit_results

,n_matches,accuracy,log_loss,brier,auc
train,4617.0,0.710201,0.571266,0.193352,0.785219
val,925.0,0.716757,0.592464,0.199154,0.773786
test,3770.0,0.644032,0.626188,0.216104,0.713937


In [388]:
comparison = pd.DataFrame({
    "GNN_accuracy": metrics_by_phase["accuracy"],
    "GNN_log_loss": metrics_by_phase["logloss"],
    "GNN_brier": metrics_by_phase["brier"],

    "BscoreLogit_accuracy": bscore_logit_results["accuracy"],
    "BscoreLogit_log_loss": bscore_logit_results["log_loss"],
    "BscoreLogit_brier": bscore_logit_results["brier"],
    "BscoreLogit_auc": bscore_logit_results["auc"],
})

comparison

,GNN_accuracy,GNN_log_loss,GNN_brier,BscoreLogit_accuracy,BscoreLogit_log_loss,BscoreLogit_brier,BscoreLogit_auc
test,0.659947,0.631730,0.217822,0.644032,0.626188,0.216104,0.713937
train,0.662195,0.596825,0.205822,0.710201,0.571266,0.193352,0.785219
val,0.700465,0.571368,0.193055,0.716757,0.592464,0.199154,0.773786


In [389]:
test_predictions = predictions_df[
    predictions_df["phase"] == "test"
].copy()

test_predictions["x_bscore_diff_scaled"] = (
    test_predictions["bscore_diff"] - mu
) / sigma

test_predictions["bscore_prob"] = bscore_logit.predict_proba(
    test_predictions[["x_bscore_diff_scaled"]]
)[:, 1]

test_predictions["bscore_pred"] = (
    test_predictions["bscore_prob"] >= 0.5
).astype(int)

In [390]:
def metrics_by_level(df):
    y = df["y_true"].to_numpy()

    gnn_prob = np.clip(
        df["prob_before"].to_numpy(),
        1e-7,
        1 - 1e-7
    )
    gnn_pred = df["pred_before"].to_numpy()

    bscore_prob = np.clip(
        df["bscore_prob"].to_numpy(),
        1e-7,
        1 - 1e-7
    )
    bscore_pred = df["bscore_pred"].to_numpy()

    return pd.Series({
        "n_matches": len(df),

        "gnn_accuracy": accuracy_score(y, gnn_pred),
        "bscore_accuracy": accuracy_score(y, bscore_pred),

        "gnn_log_loss": log_loss(y, gnn_prob, labels=[0, 1]),
        "bscore_log_loss": log_loss(
            y,
            bscore_prob,
            labels=[0, 1]
        ),

        "gnn_brier": brier_score_loss(y, gnn_prob),
        "bscore_brier": brier_score_loss(
            y,
            bscore_prob
        )
    })

comparison_by_intransitivity = (
    test_predictions
    .groupby("intransitivity_level")
    .apply(metrics_by_level)
    .reindex(["low", "medium", "high"])
)

comparison_by_intransitivity

,n_matches,gnn_accuracy,bscore_accuracy,gnn_log_loss,bscore_log_loss,gnn_brier,bscore_brier
intransitivity_level,,,,,,,
low,857.0,0.668611,0.662777,0.623726,0.618221,0.213927,0.213163
medium,914.0,0.665208,0.650985,0.643237,0.610236,0.221561,0.211389
high,1048.0,0.665076,0.701336,0.624037,0.608284,0.214387,0.202329


In [391]:
from math import erf, sqrt

try:
    from scipy.stats import t as student_t
except Exception:
    student_t = None


def diebold_mariano_test(loss_model_1, loss_model_2, h=1, alternative="two-sided"):
    loss_model_1 = np.asarray(loss_model_1, dtype=float)
    loss_model_2 = np.asarray(loss_model_2, dtype=float)

    if loss_model_1.shape != loss_model_2.shape:
        raise ValueError("Le due serie di loss devono avere la stessa lunghezza.")

    d = loss_model_1 - loss_model_2
    d = d[np.isfinite(d)]
    T = len(d)

    if T <= h:
        raise ValueError("Troppi pochi dati per il Diebold-Mariano test con questo orizzonte h.")

    d_bar = d.mean()
    centered = d - d_bar

    gamma0 = np.dot(centered, centered) / T
    long_run_var = gamma0

    for lag in range(1, h):
        gamma = np.dot(centered[lag:], centered[:-lag]) / T
        long_run_var += 2 * (1 - lag / h) * gamma

    if long_run_var <= 0:
        raise ValueError("La long-run variance non e' positiva; impossibile calcolare il test.")

    dm_stat = d_bar / np.sqrt(long_run_var / T)

    harvey_adj = np.sqrt((T + 1 - 2 * h + (h * (h - 1)) / T) / T)
    dm_stat *= harvey_adj

    if student_t is not None:
        if alternative == "two-sided":
            p_value = 2 * student_t.sf(abs(dm_stat), df=T - 1)
        elif alternative == "less":
            p_value = student_t.cdf(dm_stat, df=T - 1)
        elif alternative == "greater":
            p_value = student_t.sf(dm_stat, df=T - 1)
        else:
            raise ValueError("alternative deve essere 'two-sided', 'less' o 'greater'.")
    else:
        normal_cdf = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
        if alternative == "two-sided":
            p_value = 2 * (1 - normal_cdf(abs(dm_stat)))
        elif alternative == "less":
            p_value = normal_cdf(dm_stat)
        elif alternative == "greater":
            p_value = 1 - normal_cdf(dm_stat)
        else:
            raise ValueError("alternative deve essere 'two-sided', 'less' o 'greater'.")

    return pd.Series({
        "n_matches": T,
        "mean_loss_diff": d_bar,
        "dm_stat": dm_stat,
        "p_value": p_value
    })


test_dm = predictions_df[predictions_df["phase"] == "test"].copy()

X_test_bscore = pd.DataFrame({
    "x_bscore_diff_scaled": (test_dm["bscore_diff"] - mu) / sigma
})

eps = 1e-7
y_test_dm = test_dm["y_true"].to_numpy()
gnn_prob = np.clip(test_dm["prob_before"].to_numpy(), eps, 1 - eps)
bscore_prob = np.clip(bscore_logit.predict_proba(X_test_bscore)[:, 1], eps, 1 - eps)

gnn_logloss = -(y_test_dm * np.log(gnn_prob) + (1 - y_test_dm) * np.log(1 - gnn_prob))
bscore_logloss = -(y_test_dm * np.log(bscore_prob) + (1 - y_test_dm) * np.log(1 - bscore_prob))

gnn_brier = (gnn_prob - y_test_dm) ** 2
bscore_brier = (bscore_prob - y_test_dm) ** 2

dm_results = pd.DataFrame({
    "log_loss": diebold_mariano_test(gnn_logloss, bscore_logloss),
    "brier": diebold_mariano_test(gnn_brier, bscore_brier)
}).T

dm_results.index.name = "loss"
dm_results

,n_matches,mean_loss_diff,dm_stat,p_value
loss,,,,
log_loss,3770.0,0.005542,0.800397,0.423531
brier,3770.0,0.001719,0.698585,0.484855
